# Banco Agrícola (EntropyHack) — Sistema de Alerta Temprana de Morosidad Preventiva
## Arquitectura Modular de Machine Learning: EDA, Limpieza, Feature Engineering y Comparación de Modelos

### Objetivo del Notebook:
Construir un motor preventivo y explicable de riesgo crediticio alineado a la normativa salvadoreña de la **Superintendencia del Sistema Financiero (NCB-022 / NASF-09)**. El sistema predice la severidad temporal del atraso antes de que el cliente caiga en mora irreversible.

---
### Misión 1: Configuración del Entorno, Librerías y Parámetros Globales
Importamos las dependencias necesarias, definimos la semilla de reproducibilidad y configuramos el estilo visual con la identidad de **Banco Agrícola**.


In [ ]:
import os
import time
import warnings
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import kagglehub

# Modelos y Validación
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
import shap

# Parámetros de Configuración Global
SEMILLA_ALEATORIA = 42
NUMERO_DE_FOLDS_VALIDACION = 5
warnings.filterwarnings('ignore')

# Configuración Estética (Identidad Banco Agrícola / Grupo Bancolombia)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

print("Entorno configurado correctamente con todas las dependencias.")


### Misión 2: Ingesta Modular de Datos con KaggleHub
Utilizamos `kagglehub.competition_download` para resolver la ruta del dataset. Si ya está descargado en tu equipo, **kagglehub lo detecta en caché en menos de 1 segundo sin volver a descargarlo**; si el notebook es clonado por otro integrante del equipo, lo descargará automáticamente.


In [ ]:
# 1. Resolución y descarga automática del dataset con KaggleHub
print("Resolviendo dataset desde KaggleHub...")
ruta_directorio_datos = kagglehub.competition_download('home-credit-default-risk')
print("Ruta de los archivos de la competencia:", ruta_directorio_datos)

def cargar_conjuntos_de_datos(ruta_directorio: str):
    """
    Carga las tres fuentes primarias para el entrenamiento del sistema:
    1. application_train.csv (Perfil socioeconómico y crediticio)
    2. installments_payments.csv (Historial transaccional cuota por cuota)
    3. bureau.csv (Historial de créditos vigentes en el sistema financiero)
    """
    print("Iniciando carga de archivos CSV...")
    tiempo_inicio = time.time()
    
    tabla_solicitudes = pd.read_csv(os.path.join(ruta_directorio, 'application_train.csv'))
    tabla_pagos_cuotas = pd.read_csv(os.path.join(ruta_directorio, 'installments_payments.csv'))
    
    columnas_buro_necesarias = ['SK_ID_CURR', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT']
    tabla_buro_credito = pd.read_csv(
        os.path.join(ruta_directorio, 'bureau.csv'),
        usecols=columnas_buro_necesarias
    )
    
    duracion = time.time() - tiempo_inicio
    print(f"Carga completada en {duracion:.2f} segundos.")
    print(f"• Solicitudes: {tabla_solicitudes.shape[0]:,} filas x {tabla_solicitudes.shape[1]} columnas")
    print(f"• Cuotas: {tabla_pagos_cuotas.shape[0]:,} filas x {tabla_pagos_cuotas.shape[1]} columnas")
    print(f"• Buró de Crédito: {tabla_buro_credito.shape[0]:,} filas x {tabla_buro_credito.shape[1]} columnas")
    
    return tabla_solicitudes, tabla_pagos_cuotas, tabla_buro_credito

# Cargar las tablas maestras
datos_solicitudes_crudos, datos_cuotas_crudos, datos_buro_crudos = cargar_conjuntos_de_datos(ruta_directorio_datos)


### Misión 3: Target Engineering Multiclase Normativo (SSF El Salvador)
Calculamos la severidad de mora cuota a cuota a partir de `installments_payments.csv`. Asignamos mora máxima (365 días) a impagos absolutos y categorizamos según la normativa de la **Superintendencia del Sistema Financiero de El Salvador**:
- `Healthy` ($\le 0$ días): Al día o con pago anticipado.
- `A1` (1 a 14 días): Alerta muy temprana.
- `A2` (15 a 30 días): Estrés financiero moderado (reprogramación preventiva).
- `B` (31 a 60 días): Riesgo potencial de impago.
- `C` (61 a 120 días): Mora severa.
- `D_E` (121 a 365 días): Pre-cobranza judicial / castigo.


In [ ]:
def clasificar_tramo_riesgo_ssf(dias_de_atraso: float) -> str:
    """Mapea los días de atraso a la categoría normativa de la SSF."""
    if dias_de_atraso <= 0:
        return 'Healthy'
    elif dias_de_atraso <= 14:
        return 'A1'
    elif dias_de_atraso <= 30:
        return 'A2'
    elif dias_de_atraso <= 60:
        return 'B'
    elif dias_de_atraso <= 120:
        return 'C'
    else:
        return 'D_E'

def construir_target_normativo_ssf(tabla_cuotas: pd.DataFrame) -> pd.DataFrame:
    """
    Construye el objetivo multiclase analizando el historial completo de pagos.
    Maneja rigurosamente los impagos totales (sin registro de abono) asignando 365 días.
    """
    print("Calculando días de mora y target normativo SSF...")
    cuotas_procesadas = tabla_cuotas.copy()
    
    # Días de atraso = Fecha de pago real - Fecha de vencimiento pactada
    cuotas_procesadas['dias_atraso_calculados'] = (
        cuotas_procesadas['DAYS_ENTRY_PAYMENT'] - cuotas_procesadas['DAYS_INSTALMENT']
    )
    
    # Tratamiento de impago total: Si no hay fecha de pago, la mora es máxima
    mascara_sin_pago = cuotas_procesadas['DAYS_ENTRY_PAYMENT'].isnull()
    cuotas_procesadas['dias_atraso_limpios'] = cuotas_procesadas['dias_atraso_calculados'].clip(upper=365)
    cuotas_procesadas.loc[mascara_sin_pago, 'dias_atraso_limpios'] = 365.0
    cuotas_procesadas.loc[cuotas_procesadas['AMT_PAYMENT'].isnull(), 'AMT_PAYMENT'] = 0.0
    
    # Obtener el peor atraso histórico por cliente
    resumen_morosidad = cuotas_procesadas.groupby('SK_ID_CURR')['dias_atraso_limpios'].max().reset_index()
    resumen_morosidad.rename(columns={'dias_atraso_limpios': 'maximo_atraso_dias_historico'}, inplace=True)
    
    # Asignación de categoría SSF
    resumen_morosidad['clase_morosidad_ssf'] = resumen_morosidad['maximo_atraso_dias_historico'].apply(clasificar_tramo_riesgo_ssf)
    
    return resumen_morosidad

# Ejecución de la Misión 3
tabla_target_clientes = construir_target_normativo_ssf(datos_cuotas_crudos)

# Visualización de la distribución del target
distribucion_clases = tabla_target_clientes['clase_morosidad_ssf'].value_counts(normalize=True) * 100
plt.figure(figsize=(9, 4))
paleta_riesgo = ['#10b981', '#3b82f6', '#f59e0b', '#f97316', '#ef4444', '#7f1d1d']
sns.barplot(x=distribucion_clases.index, y=distribucion_clases.values, palette=paleta_riesgo)
plt.title("Distribución de Clientes por Categoría de Riesgo Preventivo SSF", fontweight='bold')
plt.xlabel("Categoría SSF")
plt.ylabel("Porcentaje de Clientes (%)")
for indice, valor in enumerate(distribucion_clases.values):
    plt.text(indice, valor + 0.7, f"{valor:.1f}%", ha='center', fontweight='bold')
plt.ylim(0, 55)
plt.tight_layout()
plt.show()


### Misión 4: Limpieza de Datos Modular (Data Cleaning)
Implementamos la función `limpiar_datos_solicitudes` aplicando las decisiones acordadas:
1. Filtrado del outlier tipográfico individual de \$117M (`AMT_INCOME_TOTAL < 50_000_000`).
2. Reemplazo del valor centinela `DAYS_EMPLOYED == 365243` por `np.nan` y creación del flag `dias_empleado_anomalia = 1`.
3. Imputación de la moda `'F'` a los 4 registros con `CODE_GENDER == 'XNA'`.
4. Transformación de `DAYS_BIRTH` a años enteros positivos (`edad_en_anios`).
5. Eliminación de las 47 variables inmobiliarias ruidosas (`*_AVG`, `*_MODE`, `*_MEDI`).
6. Preservación intacta de `EXT_SOURCE_1`, `EXT_SOURCE_2` y `EXT_SOURCE_3`.


In [ ]:
def limpiar_datos_solicitudes(tabla_solicitudes: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica el pipeline integral de limpieza sobre la tabla de solicitudes:
    - Remoción de outliers extremos de ingresos.
    - Corrección de la anomalía de empleo de 1000 años.
    - Limpieza categórica de género.
    - Transformación de edad a número entero positivo.
    - Poda de variables inmobiliarias con alta ausencia.
    """
    print("Iniciando limpieza de datos de solicitudes...")
    solicitudes_limpias = tabla_solicitudes.copy()
    
    # 1. Filtrado de outlier de ingresos millonarios atípicos
    cantidad_inicial = len(solicitudes_limpias)
    solicitudes_limpias = solicitudes_limpias[solicitudes_limpias['AMT_INCOME_TOTAL'] < 50_000_000].copy()
    print(f"• Registros filtrados por outlier de ingresos: {cantidad_inicial - len(solicitudes_limpias)}")
    
    # 2. Corrección de anomalía en antigüedad laboral (365243 días)
    solicitudes_limpias['dias_empleado_anomalia'] = (solicitudes_limpias['DAYS_EMPLOYED'] == 365243).astype(int)
    solicitudes_limpias['DAYS_EMPLOYED'] = solicitudes_limpias['DAYS_EMPLOYED'].replace(365243, np.nan)
    
    # 3. Corrección de género (imputar moda 'F' a registros 'XNA')
    moda_genero = solicitudes_limpias['CODE_GENDER'].mode()[0]
    solicitudes_limpias['CODE_GENDER'] = solicitudes_limpias['CODE_GENDER'].replace('XNA', moda_genero)
    
    # 4. Edad como entero positivo (int)
    solicitudes_limpias['edad_en_anios'] = (-solicitudes_limpias['DAYS_BIRTH'] // 365).astype(int)
    
    # 5. Eliminación de las 47 variables inmobiliarias
    prefijos_inmobiliarios = [
        'APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD',
        'COMMONAREA', 'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN',
        'LANDAREA', 'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS',
        'NONLIVINGAREA', 'FONDKAPREMONT', 'HOUSETYPE', 'TOTALAREA',
        'WALLSMATERIAL', 'EMERGENCYSTATE'
    ]
    columnas_inmobiliarias = [
        columna for columna in solicitudes_limpias.columns 
        if any(prefijo in columna for prefijo in prefijos_inmobiliarios)
    ]
    solicitudes_limpias.drop(columns=columnas_inmobiliarias, inplace=True, errors='ignore')
    print(f"• Variables inmobiliarias eliminadas: {len(columnas_inmobiliarias)}")
    
    print(f"Dimensiones de solicitudes tras limpieza: {solicitudes_limpias.shape}")
    return solicitudes_limpias

# Ejecución de la Misión 4
datos_solicitudes_limpios = limpiar_datos_solicitudes(datos_solicitudes_crudos)


### Misión 5: Feature Engineering — Ratios Financieros Ganadores
Construimos los ratios financieros clave demostrados en la competencia de Kaggle para modelar la capacidad de pago y el apalancamiento:
- `credit_annuity_ratio`: Duración relativa de la deuda (`AMT_CREDIT / AMT_ANNUITY`).
- `credit_goods_price_ratio`: Apalancamiento respecto al bien adquirido (`AMT_CREDIT / AMT_GOODS_PRICE`).
- `credit_downpayment`: Margen o prima aportada (`AMT_GOODS_PRICE - AMT_CREDIT`).
- `debt_to_income_ratio`: Carga financiera respecto a los ingresos (`AMT_CREDIT / AMT_INCOME_TOTAL`).
- `annuity_income_ratio`: Proporción del salario mensual consumido por la cuota (`AMT_ANNUITY / AMT_INCOME_TOTAL`).
- `debt_credit_ratio`: Grado de endeudamiento vigente en el sistema financiero (integrado desde `bureau.csv`).


In [ ]:
def construir_ratios_financieros(
    tabla_solicitudes: pd.DataFrame,
    tabla_buro: pd.DataFrame
) -> pd.DataFrame:
    """
    Calcula los ratios de solvencia y apalancamiento a partir de la solicitud
    y del historial consolidado de créditos vigentes en el Buró.
    """
    print("Calculando ratios financieros inteligentes...")
    solicitudes_con_ratios = tabla_solicitudes.copy()
    
    # Ratios de Solicitud
    solicitudes_con_ratios['credit_annuity_ratio'] = (
        solicitudes_con_ratios['AMT_CREDIT'] / (solicitudes_con_ratios['AMT_ANNUITY'] + 1e-5)
    )
    solicitudes_con_ratios['credit_goods_price_ratio'] = (
        solicitudes_con_ratios['AMT_CREDIT'] / (solicitudes_con_ratios['AMT_GOODS_PRICE'] + 1e-5)
    )
    solicitudes_con_ratios['credit_downpayment'] = (
        solicitudes_con_ratios['AMT_GOODS_PRICE'] - solicitudes_con_ratios['AMT_CREDIT']
    )
    solicitudes_con_ratios['debt_to_income_ratio'] = (
        solicitudes_con_ratios['AMT_CREDIT'] / (solicitudes_con_ratios['AMT_INCOME_TOTAL'] + 1e-5)
    )
    solicitudes_con_ratios['annuity_income_ratio'] = (
        solicitudes_con_ratios['AMT_ANNUITY'] / (solicitudes_con_ratios['AMT_INCOME_TOTAL'] + 1e-5)
    )
    
    # Integración con Buró de Crédito para debt_credit_ratio
    resumen_buro = tabla_buro.groupby('SK_ID_CURR').agg({
        'AMT_CREDIT_SUM': 'sum',
        'AMT_CREDIT_SUM_DEBT': 'sum'
    }).reset_index()
    
    resumen_buro['debt_credit_ratio'] = (
        resumen_buro['AMT_CREDIT_SUM_DEBT'] / (resumen_buro['AMT_CREDIT_SUM'] + 1e-5)
    )
    
    # Merge con solicitudes
    solicitudes_con_ratios = solicitudes_con_ratios.merge(
        resumen_buro[['SK_ID_CURR', 'debt_credit_ratio']],
        on='SK_ID_CURR',
        how='left'
    )
    
    print("Ratios financieros integrados con éxito.")
    return solicitudes_con_ratios

# Ejecución de la Misión 5
datos_solicitudes_con_ratios = construir_ratios_financieros(datos_solicitudes_limpios, datos_buro_crudos)


### Misión 6: Feature Engineering — Agregaciones de Recencia y Subpago
Para advertir el riesgo preventivo **15 a 45 días antes**, construimos variables temporales segmentadas en ventanas retrospectivas (`DAYS_INSTALMENT >= -W`) para 60, 90, 180 y 365 días:
- Días de mora: `mean`, `max`, `sum`.
- Diferencia de pago (`AMT_PAYMENT - AMT_INSTALMENT`): `mean`, `min`, `sum`.
- KPI global de Underpayment: `installment_payment_ratio` (media histórica de pagos por debajo de la cuota).


In [ ]:
def construir_metricas_recencia_cuotas(tabla_cuotas: pd.DataFrame) -> pd.DataFrame:
    """
    Genera agregaciones de recencia en bloques de tiempo (60, 90, 180, 365 días)
    y KPIs de comportamiento de pago incompleto (underpayment).
    """
    print("Calculando agregaciones de recencia temporal y KPIs de underpayment...")
    cuotas = tabla_cuotas.copy()
    
    # Cálculo de atraso y diferencia de pago
    cuotas['dias_de_atraso'] = cuotas['DAYS_ENTRY_PAYMENT'] - cuotas['DAYS_INSTALMENT']
    cuotas['dias_atraso_limpios'] = cuotas['dias_de_atraso'].clip(upper=365)
    cuotas.loc[cuotas['DAYS_ENTRY_PAYMENT'].isnull(), 'dias_atraso_limpios'] = 365.0
    cuotas.loc[cuotas['AMT_PAYMENT'].isnull(), 'AMT_PAYMENT'] = 0.0
    
    # Diferencia de pago (negativo = subpago o pago insuficiente)
    cuotas['diferencia_de_pago'] = cuotas['AMT_PAYMENT'] - cuotas['AMT_INSTALMENT']
    
    # KPIs Globales de Underpayment
    kpis_globales_cuotas = cuotas.groupby('SK_ID_CURR').agg(
        installment_payment_ratio=('diferencia_de_pago', 'mean'),
        max_monto_cuota_historico=('AMT_INSTALMENT', 'max')
    ).reset_index()
    
    lista_dataframes_ventanas = [kpis_globales_cuotas]
    
    # Agregaciones en Bloques Temporales (60, 90, 180 y 365 días)
    ventanas_en_dias = [60, 90, 180, 365]
    for dias_ventana in ventanas_en_dias:
        # Filtrar cuotas ocurridas en los últimos 'dias_ventana' días
        subconjunto_ventana = cuotas[cuotas['DAYS_INSTALMENT'] >= -dias_ventana]
        
        agregado_ventana = subconjunto_ventana.groupby('SK_ID_CURR').agg({
            'dias_atraso_limpios': ['mean', 'max', 'sum'],
            'diferencia_de_pago': ['mean', 'min', 'sum']
        })
        
        # Nombres de columnas limpios y descriptivos
        agregado_ventana.columns = [
            f"{metrica}_{estadistico}_{dias_ventana}dias" 
            for metrica, estadistico in agregado_ventana.columns
        ]
        agregado_ventana.reset_index(inplace=True)
        lista_dataframes_ventanas.append(agregado_ventana)
    
    # Unificación de todas las ventanas por cliente
    caracteristicas_transaccionales = reduce(
        lambda izq, der: pd.merge(izq, der, on='SK_ID_CURR', how='left'),
        lista_dataframes_ventanas
    )
    
    print(f"Características de recencia generadas: {caracteristicas_transaccionales.shape[1] - 1}")
    return caracteristicas_transaccionales

# Ejecución de la Misión 6
caracteristicas_cuotas_recencia = construir_metricas_recencia_cuotas(datos_cuotas_crudos)


### Misión 7: Consolidación y Preparación de la Matriz de Modelado
Unificamos las solicitudes con ratios, las características de recencia transaccional y el target multiclase. Codificamos las variables categóricas restantes y preparamos la matriz predictiva final `X` y el vector objetivo `y`.


In [ ]:
def preparar_matriz_final_modelado(
    tabla_solicitudes_con_ratios: pd.DataFrame,
    tabla_caracteristicas_cuotas: pd.DataFrame,
    tabla_target: pd.DataFrame
):
    """
    Consolida todas las fuentes, genera el ratio annuity_to_max_installment_ratio,
    aplica codificación a variables categóricas y separa X e y.
    """
    print("Consolidando dataset final para entrenamiento...")
    
    # 1. Cruce con el target normativo y las variables transaccionales
    matriz_unificada = tabla_solicitudes_con_ratios.merge(
        tabla_target[['SK_ID_CURR', 'clase_morosidad_ssf']],
        on='SK_ID_CURR',
        how='inner'
    )
    matriz_unificada = matriz_unificada.merge(
        tabla_caracteristicas_cuotas,
        on='SK_ID_CURR',
        how='left'
    )
    
    # 2. Ratio final annuity_to_max_installment_ratio
    matriz_unificada['annuity_to_max_installment_ratio'] = (
        matriz_unificada['AMT_ANNUITY'] / (matriz_unificada['max_monto_cuota_historico'] + 1e-5)
    )
    
    # 3. Separación de etiquetas y mapeo multiclase
    etiquetas_clases = ['Healthy', 'A1', 'A2', 'B', 'C', 'D_E']
    mapeo_clases_a_enteros = {nombre_clase: indice for indice, nombre_clase in enumerate(etiquetas_clases)}
    mapeo_enteros_a_clases = {indice: nombre_clase for nombre_clase, indice in mapeo_clases_a_enteros.items()}
    
    vector_y = matriz_unificada['clase_morosidad_ssf'].map(mapeo_clases_a_enteros).astype(int)
    
    # 4. Selección y codificación de columnas predictoras
    columnas_a_descartar = ['SK_ID_CURR', 'TARGET', 'clase_morosidad_ssf']
    columnas_predictoras = [col for col in matriz_unificada.columns if col not in columnas_a_descartar]
    
    matriz_X = matriz_unificada[columnas_predictoras].copy()
    
    # Codificación de variables categóricas con One-Hot Encoding limpio
    columnas_categoricas = matriz_X.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"• Codificando {len(columnas_categoricas)} variables categóricas...")
    matriz_X = pd.get_dummies(matriz_X, columns=columnas_categoricas, drop_first=True)
    
    # Limpieza de caracteres especiales en nombres de columnas para compatibilidad
    matriz_X.columns = [
        columna.replace(' ', '_').replace(':', '_').replace('-', '_').replace(',', '_')
        for columna in matriz_X.columns
    ]
    
    print(f"Matriz predictiva X lista: {matriz_X.shape[0]:,} clientes x {matriz_X.shape[1]} características")
    print(f"Vector objetivo y listo: {len(vector_y):,} etiquetas multiclase")
    
    return matriz_X, vector_y, mapeo_enteros_a_clases

# Ejecución de la Misión 7
matriz_caracteristicas_X, vector_etiquetas_y, diccionario_clases = preparar_matriz_final_modelado(
    datos_solicitudes_con_ratios,
    caracteristicas_cuotas_recencia,
    tabla_target_clientes
)


### Misión 8: Motor Modular de Entrenamiento y Validación Cruzada
Definimos la función general y reutilizable `entrenar_y_evaluar_modelo`. Esta función:
- Aplica validación cruzada con `StratifiedKFold` (5 pliegues).
- Admite un parámetro opcional de muestreo (`tamano_muestra_evaluacion`) para iteraciones rápidas durante el hackathon.
- Soporta modelos nativos con nulos (LightGBM, XGBoost, CatBoost) y modelos que requieren imputación previa (Random Forest).
- Retorna un diccionario estructurado con métricas de desempeño multiclase (**Balanced Accuracy**, **Macro F1-Score**, **Weighted F1-Score** y **Multiclass ROC-AUC**).


In [ ]:
def entrenar_y_evaluar_modelo(
    nombre_del_modelo: str,
    estimador_modelo,
    matriz_X: pd.DataFrame,
    vector_y: pd.Series,
    numero_de_folds: int = NUMERO_DE_FOLDS_VALIDACION,
    requiere_imputacion: bool = False,
    tamano_muestra_evaluacion: int = 50000
) -> dict:
    """
    Función genérica y reutilizable para entrenar y evaluar cualquier modelo multiclase
    utilizando validación cruzada estratificada (Stratified K-Fold).
    """
    print(f"\n{'='*70}")
    print(f"Iniciando entrenamiento y validación cruzada para: {nombre_del_modelo}")
    print(f"{'='*70}")
    
    # Submuestreo estratificado para agilidad en fase de experimentación si se especifica
    if tamano_muestra_evaluacion is not None and len(matriz_X) > tamano_muestra_evaluacion:
        print(f"• Utilizando submuestra estratificada de {tamano_muestra_evaluacion:,} registros para agilidad...")
        indice_muestra = vector_y.groupby(vector_y, group_keys=False).apply(
            lambda s: s.sample(int(np.rint(tamano_muestra_evaluacion * len(s) / len(vector_y))), random_state=SEMILLA_ALEATORIA)
        ).index
        X_trabajo = matriz_X.loc[indice_muestra].reset_index(drop=True)
        y_trabajo = vector_y.loc[indice_muestra].reset_index(drop=True)
    else:
        X_trabajo = matriz_X.reset_index(drop=True)
        y_trabajo = vector_y.reset_index(drop=True)
        
    validador_estratificado = StratifiedKFold(
        n_splits=numero_de_folds,
        shuffle=True,
        random_state=SEMILLA_ALEATORIA
    )
    
    lista_accuracy = []
    lista_balanced_accuracy = []
    lista_f1_macro = []
    lista_f1_weighted = []
    lista_roc_auc_multiclase = []
    
    ultimo_modelo_entrenado = None
    ultimo_imputador = None
    tiempo_total_inicio = time.time()
    
    for numero_fold, (indices_entrenamiento, indices_validacion) in enumerate(validador_estratificado.split(X_trabajo, y_trabajo), 1):
        X_entrenamiento = X_trabajo.iloc[indices_entrenamiento].copy()
        y_entrenamiento = y_trabajo.iloc[indices_entrenamiento].copy()
        X_validacion = X_trabajo.iloc[indices_validacion].copy()
        y_validacion = y_trabajo.iloc[indices_validacion].copy()
        
        # Imputación de nulos si el estimador no los maneja nativamente (ej. Random Forest)
        if requiere_imputacion:
            imputador = SimpleImputer(strategy='median')
            X_entrenamiento = pd.DataFrame(imputador.fit_transform(X_entrenamiento), columns=X_entrenamiento.columns)
            X_validacion = pd.DataFrame(imputador.transform(X_validacion), columns=X_validacion.columns)
            ultimo_imputador = imputador
            
        # Ajuste del modelo
        tiempo_fold = time.time()
        estimador_modelo.fit(X_entrenamiento, y_entrenamiento)
        duracion_fold = time.time() - tiempo_fold
        
        # Predicciones
        predicciones_clase = estimador_modelo.predict(X_validacion)
        if hasattr(predicciones_clase, 'ndim') and predicciones_clase.ndim > 1:
            predicciones_clase = predicciones_clase.ravel()
            
        probabilidades_predichas = estimador_modelo.predict_proba(X_validacion)
        
        # Cálculo de métricas
        acc = accuracy_score(y_validacion, predicciones_clase)
        b_acc = balanced_accuracy_score(y_validacion, predicciones_clase)
        f1_m = f1_score(y_validacion, predicciones_clase, average='macro', zero_division=0)
        f1_w = f1_score(y_validacion, predicciones_clase, average='weighted', zero_division=0)
        
        try:
            auc = roc_auc_score(y_validacion, probabilidades_predichas, multi_class='ovr', average='weighted')
        except Exception:
            auc = np.nan
            
        lista_accuracy.append(acc)
        lista_balanced_accuracy.append(b_acc)
        lista_f1_macro.append(f1_m)
        lista_f1_weighted.append(f1_w)
        lista_roc_auc_multiclase.append(auc)
        
        print(f"  [Fold {numero_fold}/{numero_de_folds}] Duración: {duracion_fold:.1f}s | Acc: {acc:.4f} | Balanced Acc: {b_acc:.4f} | F1 Macro: {f1_m:.4f} | ROC-AUC: {auc:.4f}")
        ultimo_modelo_entrenado = estimador_modelo
        
    duracion_total = time.time() - tiempo_total_inicio
    
    resultados_modelo = {
        'nombre_modelo': nombre_del_modelo,
        'accuracy_promedio': np.nanmean(lista_accuracy),
        'balanced_accuracy_promedio': np.nanmean(lista_balanced_accuracy),
        'f1_macro_promedio': np.nanmean(lista_f1_macro),
        'f1_weighted_promedio': np.nanmean(lista_f1_weighted),
        'roc_auc_promedio': np.nanmean(lista_roc_auc_multiclase),
        'tiempo_total_segundos': duracion_total,
        'modelo_entrenado': ultimo_modelo_entrenado,
        'imputador_asociado': ultimo_imputador,
        'X_muestra_validacion': X_validacion,
        'y_muestra_validacion': y_validacion
    }
    
    print(f"\nResumen {nombre_del_modelo}: Balanced Acc = {resultados_modelo['balanced_accuracy_promedio']:.4f} | ROC-AUC = {resultados_modelo['roc_auc_promedio']:.4f} | F1 Weighted = {resultados_modelo['f1_weighted_promedio']:.4f}")
    return resultados_modelo


### Misión 9: Ejecución Comparativa de los 4 Modelos
Entrenamos y comparamos los 4 modelos seleccionados mediante llamadas modulares a la función genérica:
1. **LightGBM**: Alta velocidad de cómputo, splits por hoja y manejo nativo de datos ausentes.
2. **XGBoost**: Regularización L1/L2 avanzada para control de sobreajuste.
3. **CatBoost**: Manejo sobresaliente de interacciones y splits robustos en datos tabulares.
4. **Random Forest**: Ensamble de embolsado (*bagging*), resistente a la varianza y con árboles desacoplados.


In [ ]:
# Diccionario para almacenar los resultados de los 4 modelos
registro_resultados_modelos = []

# ------------------------------------------------------------------------------
# 1. Entrenar LightGBM
# ------------------------------------------------------------------------------
configuracion_lightgbm = LGBMClassifier(
    objective='multiclass',
    num_class=6,
    learning_rate=0.03,
    n_estimators=150,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEMILLA_ALEATORIA,
    n_jobs=-1,
    verbose=-1
)
resultado_lightgbm = entrenar_y_evaluar_modelo(
    nombre_del_modelo="LightGBM",
    estimador_modelo=configuracion_lightgbm,
    matriz_X=matriz_caracteristicas_X,
    vector_y=vector_etiquetas_y,
    requiere_imputacion=False
)
registro_resultados_modelos.append(resultado_lightgbm)

# ------------------------------------------------------------------------------
# 2. Entrenar XGBoost
# ------------------------------------------------------------------------------
configuracion_xgboost = XGBClassifier(
    objective='multi:softprob',
    num_class=6,
    learning_rate=0.03,
    n_estimators=150,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEMILLA_ALEATORIA,
    n_jobs=-1,
    verbosity=0
)
resultado_xgboost = entrenar_y_evaluar_modelo(
    nombre_del_modelo="XGBoost",
    estimador_modelo=configuracion_xgboost,
    matriz_X=matriz_caracteristicas_X,
    vector_y=vector_etiquetas_y,
    requiere_imputacion=False
)
registro_resultados_modelos.append(resultado_xgboost)

# ------------------------------------------------------------------------------
# 3. Entrenar CatBoost
# ------------------------------------------------------------------------------
configuracion_catboost = CatBoostClassifier(
    loss_function='MultiClass',
    learning_rate=0.05,
    iterations=150,
    depth=5,
    random_seed=SEMILLA_ALEATORIA,
    verbose=0,
    thread_count=-1
)
resultado_catboost = entrenar_y_evaluar_modelo(
    nombre_del_modelo="CatBoost",
    estimador_modelo=configuracion_catboost,
    matriz_X=matriz_caracteristicas_X,
    vector_y=vector_etiquetas_y,
    requiere_imputacion=False
)
registro_resultados_modelos.append(resultado_catboost)

# ------------------------------------------------------------------------------
# 4. Entrenar Random Forest
# ------------------------------------------------------------------------------
configuracion_random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=SEMILLA_ALEATORIA,
    n_jobs=-1
)
resultado_random_forest = entrenar_y_evaluar_modelo(
    nombre_del_modelo="Random Forest",
    estimador_modelo=configuracion_random_forest,
    matriz_X=matriz_caracteristicas_X,
    vector_y=vector_etiquetas_y,
    requiere_imputacion=True
)
registro_resultados_modelos.append(resultado_random_forest)


### Misión 10: Tabla Comparativa, Explicabilidad con SHAP y Exportación
Generamos el cuadro de honor comparativo con los 4 modelos, visualizamos la importancia de variables con **SHAP TreeExplainer** y exportamos el artefacto del mejor modelo (`model_abcd.pkl`) para integrarse con la API de FastAPI y la interfaz empática de Banco Agrícola.


In [ ]:
# 1. Tabla Comparativa de Rendimiento
tabla_comparativa = pd.DataFrame([
    {
        'Modelo': r['nombre_modelo'],
        'Balanced Accuracy': f"{r['balanced_accuracy_promedio']:.4f}",
        'ROC-AUC (OVR)': f"{r['roc_auc_promedio']:.4f}",
        'F1 Macro': f"{r['f1_macro_promedio']:.4f}",
        'F1 Weighted': f"{r['f1_weighted_promedio']:.4f}",
        'Tiempo (s)': f"{r['tiempo_total_segundos']:.1f}"
    }
    for r in registro_resultados_modelos
])

print("="*75)
print("TABLA COMPARATIVA DE MODELOS MULTICLASE (SUPERINTENDENCIA SSF)")
print("="*75)
print(tabla_comparativa.to_string(index=False))

# 2. Selección del Mejor Modelo
mejor_resultado = max(registro_resultados_modelos, key=lambda x: x['roc_auc_promedio'])
mejor_modelo = mejor_resultado['modelo_entrenado']
print(f"\n🏆 Modelo Campeón Seleccionado: {mejor_resultado['nombre_modelo']} con ROC-AUC = {mejor_resultado['roc_auc_promedio']:.4f}")

# 3. Explicabilidad con SHAP
print("\nGenerando valores SHAP para explicabilidad bancaria...")
X_muestra_shap = mejor_resultado['X_muestra_validacion'].iloc[:300]

try:
    explicador_shap = shap.TreeExplainer(mejor_modelo)
    valores_shap = explicador_shap.shap_values(X_muestra_shap)
    
    # Visualización SHAP Summary Plot
    plt.figure(figsize=(10, 6))
    plt.title(f"Importancia de Factores de Riesgo con SHAP ({mejor_resultado['nombre_modelo']})", fontsize=12, fontweight='bold')
    shap.summary_plot(
        valores_shap,
        X_muestra_shap,
        class_names=list(diccionario_clases.values()),
        show=False,
        max_display=12
    )
    plt.tight_layout()
    plt.show()
except Exception as error_shap:
    print(f"Aviso al calcular SHAP detallado: {error_shap}")

# 4. Guardado del Artefacto del Modelo para Producción
ruta_artefacto = os.path.join(os.getcwd(), 'model_abcd.pkl')
artefacto_empaquetado = {
    'modelo': mejor_modelo,
    'nombre_modelo': mejor_resultado['nombre_modelo'],
    'columnas_caracteristicas': matriz_caracteristicas_X.columns.tolist(),
    'diccionario_clases': diccionario_clases,
    'imputador': mejor_resultado['imputador_asociado']
}

joblib.dump(artefacto_empaquetado, ruta_artefacto)
print(f"\nArtefacto de modelo guardado exitosamente en: {ruta_artefacto}")
